In [ ]:
pip install -q langchain-openai langgraph pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.7/571.7 kB 9.8 MB/s eta 0:00:00


In [ ]:
import os
from typing import Annotated, Literal, TypedDict
from google.colab import userdata
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from pydantic import BaseModel, Field

# ---------------------------------------------------------------------------
# 0. API Key & OpenRouter Setup (Using 'Openai' Secret Key)
# ---------------------------------------------------------------------------
try:
    openrouter_key = userdata.get("Openai")
    if not openrouter_key:
        raise ValueError("The 'Openai' secret key is empty.")
    os.environ["OPENROUTER_API_KEY"] = openrouter_key
except Exception as e:
    raise RuntimeError(
        "Please ensure the 'Openai' secret is added in Colab Secrets and notebook access is enabled."
    ) from e

SELECTED_MODEL = "meta-llama/llama-3.3-70b-instruct"
print(f"--> Using active model: '{SELECTED_MODEL}' via OpenRouter")

llm = ChatOpenAI(
    model=SELECTED_MODEL,
    api_key=openrouter_key,
    base_url="https://openrouter.ai/api/v1",
    temperature=0,
    default_headers={
        "HTTP-Referer": "https://colab.research.google.com",
        "X-Title": "LangGraph Ticket Router",
    },
)


def extract_text_safely(content) -> str:
    """Normalizes string or block-based message output into plain text."""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict) and "text" in item:
                parts.append(item["text"])
            elif isinstance(item, str):
                parts.append(item)
            else:
                parts.append(str(item))
        return "".join(parts)
    return str(content)


# ---------------------------------------------------------------------------
# 1. State & Routing Schema
# ---------------------------------------------------------------------------
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    next_node: str


class RouteDecision(BaseModel):
    destination: Literal["tech_support_agent", "account_actions_agent"] = Field(
        description="Select 'tech_support_agent' for technical/freeze bugs, or 'account_actions_agent' for refund/billing inquiries."
    )


# ---------------------------------------------------------------------------
# 2. Tool Definition
# ---------------------------------------------------------------------------
@tool
def process_refund(user_id: str, amount: float) -> str:
    """Executes a financial refund for a specific user ID."""
    return f"SUCCESS: Refund of ${amount:.2f} has been processed for User '{user_id}'."


# ---------------------------------------------------------------------------
# 3. Agent Nodes
# ---------------------------------------------------------------------------
def supervisor_agent(state: AgentState) -> dict:
    router_llm = llm.with_structured_output(RouteDecision)
    system_prompt = (
        "You are an intent routing classifier for support tickets. "
        "Classify the inquiry into the appropriate destination."
    )
    messages = [SystemMessage(content=system_prompt)] + state["messages"]

    decision = router_llm.invoke(messages)
    destination = (
        decision.destination
        if hasattr(decision, "destination")
        else decision.get("destination", END)
    )
    return {"next_node": destination}


def tech_support_agent(state: AgentState) -> dict:
    system_prompt = SystemMessage(
        content="You are a Technical Support Specialist. Provide brief, actionable troubleshooting tips."
    )
    messages = [system_prompt] + state["messages"]
    response = llm.invoke(messages)
    text_content = extract_text_safely(response.content)

    return {
        "messages": [
            AIMessage(content=f"[Tech Support | Model: {SELECTED_MODEL}]:\n{text_content}")
        ],
        "next_node": END,
    }


def account_actions_agent(state: AgentState) -> dict:
    llm_with_tools = llm.bind_tools([process_refund])
    system_prompt = SystemMessage(
        content="You are an Account Manager. Execute the refund tool when user requests mention refund amounts and user IDs."
    )
    messages = [system_prompt] + state["messages"]
    response = llm_with_tools.invoke(messages)

    if response.tool_calls:
        tool_call = response.tool_calls[0]
        tool_output = process_refund.invoke(tool_call["args"])
        final_msg = f"[Account Agent | Model: {SELECTED_MODEL} (Tool Execution)]:\n{tool_output}"
    else:
        text_content = extract_text_safely(response.content)
        final_msg = f"[Account Agent | Model: {SELECTED_MODEL}]:\n{text_content}"

    return {
        "messages": [AIMessage(content=final_msg)],
        "next_node": END,
    }


# ---------------------------------------------------------------------------
# 4. Construct LangGraph Workflow
# ---------------------------------------------------------------------------
workflow = StateGraph(AgentState)

workflow.add_node("supervisor", supervisor_agent)
workflow.add_node("tech_support_agent", tech_support_agent)
workflow.add_node("account_actions_agent", account_actions_agent)

workflow.add_edge(START, "supervisor")

workflow.add_conditional_edges(
    "supervisor",
    lambda state: state["next_node"],
    {
        "tech_support_agent": "tech_support_agent",
        "account_actions_agent": "account_actions_agent",
        END: END,
    },
)

workflow.add_edge("tech_support_agent", END)
workflow.add_edge("account_actions_agent", END)

app = workflow.compile()

# ---------------------------------------------------------------------------
# 5. Live Execution Tests
# ---------------------------------------------------------------------------
def run_demo(user_query: str):
    print(f"\n================ USER QUERY ================\n{user_query}")
    inputs = {"messages": [HumanMessage(content=user_query)]}
    result = app.invoke(inputs)
    print("\n================ SYSTEM RESPONSE ================")
    print(f"Generated via: {SELECTED_MODEL}")
    print(result["messages"][-1].content)


if __name__ == "__main__":
    run_demo(
        "How to play football"
    )
    run_demo(
        "How can i learn using katana sword"
    )

--> Using active model: 'meta-llama/llama-3.3-70b-instruct' via OpenRouter

================ USER QUERY ================
How to play football

================ SYSTEM RESPONSE ================
Generated via: meta-llama/llama-3.3-70b-instruct
[Tech Support | Model: meta-llama/llama-3.3-70b-instruct]:
To play football, follow these basic steps:

1. **Understand the objective**: Score more points than the opposing team by carrying or throwing the ball into the end zone.
2. **Know the basic positions**: Familiarize yourself with roles like quarterback (QB), running back (RB), wide receiver (WR), and lineman.
3. **Learn the downs system**: You have 4 downs (plays) to advance the ball 10 yards or score.
4. **Master basic plays**: Start with simple runs (e.g., handoff to RB) and passes (e.g., short throw to WR).
5. **Practice safety**: Wear proper gear, including a helmet, pads, and mouthguard.

For more detailed guidance, consider consulting a coach or experienced player.

================ U